# 01 — Data cleaning and source-frequency preparation

This notebook prepares the original monthly banking/macro-financial source and quarterly real GDP source. It does not interpolate GDP. The two datasets remain at their observed frequencies so that later analysis can choose an appropriate frequency without treating estimated monthly GDP as observed data.

The source CSVs contain metadata rows after the observations. Dates are parsed to identify actual data rows rather than relying on fixed row counts.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data' / 'raw' / 'data_first.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate data/raw/data_first.csv from the current working directory.')

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)


## 1. Clean monthly banking and macro-financial observations

The monthly data include policy rate, CPI, exchange rate, M2+, gold price, ROA, NPL and CAR. Missing values are retained and reported; no blanket filling is applied.

In [ ]:
monthly_raw = pd.read_csv(RAW / 'data_first.csv')
if 'Name of Series' not in monthly_raw.columns:
    raise ValueError('Monthly source is missing Name of Series.')

monthly_raw['Date'] = pd.to_datetime(monthly_raw['Name of Series'], format='%d/%m/%Y', errors='coerce')
monthly = monthly_raw.loc[monthly_raw['Date'].notna()].drop(columns=['Name of Series']).copy()
monthly = monthly.set_index('Date').sort_index()
monthly = monthly.apply(pd.to_numeric, errors='coerce')

if monthly.empty:
    raise ValueError('No valid monthly observations were parsed.')
if monthly.index.has_duplicates:
    raise ValueError('Duplicate monthly dates found.')
if not monthly.index.is_monotonic_increasing:
    raise ValueError('Monthly dates are not sorted.')

print(f'Monthly coverage: {monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}')
print(f'Number of monthly observations: {len(monthly)}')
display(monthly.isna().sum().rename('missing').to_frame())
display(monthly.tail())


## 2. Clean quarterly real GDP without changing its frequency

GDP is retained as a quarterly `PeriodIndex`. If a monthly model later needs a real-activity proxy, use a defensible monthly indicator or a formal temporal-disaggregation method; simple linear interpolation does not create observed monthly GDP.

In [ ]:
gdp_raw = pd.read_csv(RAW / 'data_first2.csv')
gdp_value = 'Gross Domestic Product (GDP), production, real'
required = {'Name of Series', gdp_value}
if not required.issubset(gdp_raw.columns):
    raise ValueError(f'GDP source must contain {sorted(required)}')

quarters = gdp_raw['Name of Series'].astype(str).str.extract(r'(\\d{4}Q[1-4])', expand=False)
values = pd.to_numeric(gdp_raw[gdp_value], errors='coerce')
gdp = pd.DataFrame({'Quarter': quarters, 'GDP_Real': values}).dropna()
gdp['Quarter'] = pd.PeriodIndex(gdp['Quarter'], freq='Q')
gdp = gdp.sort_values('Quarter').drop_duplicates('Quarter').set_index('Quarter')

if gdp.empty:
    raise ValueError('No valid quarterly GDP observations were parsed.')
if (gdp['GDP_Real'] <= 0).any():
    raise ValueError('Real GDP contains non-positive observations; inspect the source.')

print(f'Quarterly GDP coverage: {gdp.index.min()} to {gdp.index.max()}')
print(f'Number of quarterly observations: {len(gdp)}')
display(gdp.tail())


## 3. Save clean datasets and document the frequency boundary

The monthly and quarterly files are exported separately. This prevents a downstream notebook from accidentally using interpolated GDP as if it were a genuine monthly observation.

In [ ]:
monthly.to_csv(PROCESSED / 'ghana_banking_monthly_clean.csv', index_label='Date')
gdp.to_csv(PROCESSED / 'ghana_real_gdp_quarterly_clean.csv', index_label='Quarter')

print('Saved: data/processed/ghana_banking_monthly_clean.csv')
print('Saved: data/processed/ghana_real_gdp_quarterly_clean.csv')
print('No interpolation, forward fill or backward fill has been applied.')
